# Week 7 Viz

Ноутбук с визуальным разбором витрины mart

Что здесь видно
1. Как менялась средняя температура по дням
2. Как вообще распределены осадки
3. В какие дни осадков было больше всего


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use("seaborn-v0_8-whitegrid")


def detect_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "data" / "mart" / "mart.csv").exists():
            return candidate
    raise FileNotFoundError("Не удалось найти data/mart/mart.csv")


project_root = detect_project_root()
mart_path = project_root / "data" / "mart" / "mart.csv"
figures_dir = project_root / "docs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Корень проекта: {project_root}")
print(f"Путь к mart: {mart_path}")
print(f"Папка с графиками: {figures_dir}")


In [ ]:
df = pd.read_csv(mart_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Число строк:", len(df))
print("Колонки:", df.columns.tolist())
print("Диапазон дат:", df["date"].min().date(), "->", df["date"].max().date())
print(df.head())
print(df.dtypes)


## График 1 Динамика по времени

Здесь смотрю, как по дням менялась T_mean. Еще добавлено скользящее среднее за 7 дней чтобы линия была понятнее и было видно общий тренд


In [ ]:
df["T_mean_7d"] = df["T_mean"].rolling(window=7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["date"], df["T_mean"], marker="o", linewidth=1.8, color="#1f77b4", label="Средняя температура за день")
ax.plot(df["date"], df["T_mean_7d"], linewidth=2.6, color="#d62728", label="Скользящее среднее за 7 дней")

ax.set_title("Средняя температура по дням")
ax.set_xlabel("Дата")
ax.set_ylabel("Температура, °C")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=45, ha="right")
ax.legend()
fig.tight_layout()

timeseries_path = figures_dir / "week7_timeseries.png"
fig.savefig(timeseries_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"График сохранен: {timeseries_path}")


## График 2 Распределение

Тут взята метрика P_sum, тоесть сумма осадков за день. По ней можно посмотреть какие значения встречаются чаще и есть ли сильные выбросы


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df["P_sum"], bins=12, color="#2ca02c", edgecolor="black", alpha=0.85)
ax.set_title("Распределение дневных осадков")
ax.set_xlabel("Осадки за день, мм")
ax.set_ylabel("Количество дней")
fig.tight_layout()

distribution_path = figures_dir / "week7_distribution.png"
fig.savefig(distribution_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"График сохранен: {distribution_path}")


## График 3 Ранжировка

Так как в витрине только один город, здесь сравниваются не города, а дни. Взяты 7 дней, где осадков было больше всего


In [ ]:
top_precip = (
    df.nlargest(7, "P_sum")
    .sort_values("P_sum", ascending=True)
    .assign(date_label=lambda frame: frame["date"].dt.strftime("%Y-%m-%d"))
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_precip["date_label"], top_precip["P_sum"], color="#ff7f0e")
ax.set_title("Топ-7 дней по количеству осадков")
ax.set_xlabel("Осадки, мм")
ax.set_ylabel("Дата")
fig.tight_layout()

ranking_path = figures_dir / "week7_ranking.png"
fig.savefig(ranking_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"График сохранен: {ranking_path}")
top_precip[["date_label", "P_sum", "rainy_hours"]]


## Коротко по графикам

1. График 1. В начале периода температура в целом ниже, чем ближе к концу. Самый холодный день тут 2026-02-08 с T_mean = -1.77 °C, а самый теплый  2026-02-23 с T_mean = 17.59 градусов. По скользящему среднему тоже видно, что дальше температура стала выше

2. График 2. По распределению видно, что дней без осадков много. У 21 из 29 дней P_sum = 0, тоесть чаще всего день был без дождя. Но при этом есть несколько дней, где осадки уже заметно больше остальных

3. График 3. Больше всего осадков было 2026-02-25, там P_sum = 49.5 мм. Следующий по величине день 2026-02-11 с 13.5 мм, то есть отрыв довольно большой. Получается, что 2026-02-25 сильно выделяется на фоне остальных дней
